# V1-05: OpenDART 기업 식별(Entity Resolution) 기준선 비교

사용자 입력 기업명을 올바른 OpenDART `corp_code` 후보로 연결하는 방식을 비교합니다. LLM·LangChain·LangGraph·Agent는 사용하지 않습니다.

비교 대상:

1. **정확 일치 기준선**: `corp_name`이 입력과 정확히 하나만 일치할 때 선택
2. **정규화 후보**: 공백·대소문자 정규화, 제한된 별칭 사전, 상장 종목 우선

이 notebook은 최종 구현이나 최종 후보 확정이 아닙니다. 같은 개발 사례에서 두 후보의 기업 식별 품질을 확인하는 기준선 실험입니다.

## 평가 기준

- **Entity Accuracy**: 정답 상장사가 있는 사례에서 최종 선택 종목코드가 정답과 일치한 비율
- **Candidate Recall**: 선택 전 후보 집합 안에 정답 종목코드가 포함된 비율
- **Coverage**: 자동으로 기업 하나를 선택한 비율
- **Unsafe Selection Rate**: 자동 선택했지만 틀린 비율
- **보류 정확도**: 존재하지 않는 기업처럼 자동 선택하면 안 되는 사례를 올바르게 보류한 비율
- **p50/p95 지연시간**: 기업코드 목록을 메모리에 올린 뒤, 질의 하나를 반복 처리한 시간

후보가 여러 개면 억지로 선택하지 않고 `보류`로 남기는 것이 안전합니다. 따라서 Coverage만 높고 Unsafe Selection Rate가 높으면 좋은 방식이 아닙니다.

## 1. 환경과 OpenDART 기업코드 목록 로드

기업코드 ZIP은 한 번만 내려받아 메모리에서 재사용합니다. 반복 검색마다 ZIP을 다시 받지 않기 위한 실험 조건입니다.

In [1]:
import io
import json
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from statistics import median

In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

print('OpenDART API 키를 읽었습니다. 값은 출력하지 않습니다.')

OpenDART API 키를 읽었습니다. 값은 출력하지 않습니다.


In [3]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes

In [4]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]

load_started_at = time.perf_counter()
corp_records = load_corp_records(api_key)
load_elapsed_ms = (time.perf_counter() - load_started_at) * 1_000

print(f'기업코드 레코드 수: {len(corp_records):,}')
print(f'초기 ZIP 로드 시간: {load_elapsed_ms:.1f} ms')

기업코드 레코드 수: 118,831
초기 ZIP 로드 시간: 2542.7 ms


## 2. 개발용 평가 사례와 보류 사례

정답은 상장 종목코드로 둡니다. 같은 이름이 여러 개여도 상장 후보를 구분할 수 있기 때문입니다. `네이버 → NAVER` 별칭은 개발 과정에서 직접 넣은 규칙이므로, 이 사례의 점수는 별칭 일반화 성능이 아닙니다. 최종 후보를 정하기 전에는 별도 holdout 사례와 정답 출처·기준일이 필요합니다.

In [5]:
DEVELOPMENT_CASES = [
    {'유형': '정확 법인명', '질문 기업명': '삼성전자', '정답 종목코드': '005930'},
    {'유형': '정확 법인명', '질문 기업명': 'SK하이닉스', '정답 종목코드': '000660'},
    {'유형': '공백 변형', '질문 기업명': 'SK 하이닉스', '정답 종목코드': '000660'},
    {'유형': '공백 변형', '질문 기업명': '현대 자동차', '정답 종목코드': '005380'},
    {'유형': '대소문자 변형', '질문 기업명': 'naver', '정답 종목코드': '035420'},
    {'유형': '별칭', '질문 기업명': '네이버', '정답 종목코드': '035420'},
    {'유형': '동일 법인명', '질문 기업명': '셀트리온', '정답 종목코드': '068270'},
    {'유형': '정확 법인명', '질문 기업명': '리노공업', '정답 종목코드': '058470'},
]
SAFETY_CASES = [
    {'유형': '존재하지 않는 기업', '질문 기업명': 'FinSight가상기업', '정답 종목코드': None},
]

print(f'개발용 양성 사례 수: {len(DEVELOPMENT_CASES)}')
print(f'보류 사례 수: {len(SAFETY_CASES)}')
DEVELOPMENT_CASES + SAFETY_CASES

개발용 양성 사례 수: 8
보류 사례 수: 1


[{'유형': '정확 법인명', '질문 기업명': '삼성전자', '정답 종목코드': '005930'},
 {'유형': '정확 법인명', '질문 기업명': 'SK하이닉스', '정답 종목코드': '000660'},
 {'유형': '공백 변형', '질문 기업명': 'SK 하이닉스', '정답 종목코드': '000660'},
 {'유형': '공백 변형', '질문 기업명': '현대 자동차', '정답 종목코드': '005380'},
 {'유형': '대소문자 변형', '질문 기업명': 'naver', '정답 종목코드': '035420'},
 {'유형': '별칭', '질문 기업명': '네이버', '정답 종목코드': '035420'},
 {'유형': '동일 법인명', '질문 기업명': '셀트리온', '정답 종목코드': '068270'},
 {'유형': '정확 법인명', '질문 기업명': '리노공업', '정답 종목코드': '058470'},
 {'유형': '존재하지 않는 기업', '질문 기업명': 'FinSight가상기업', '정답 종목코드': None}]

## 3. 비교할 기업 식별 후보

정규화 후보의 별칭 사전은 지금은 `네이버 → NAVER` 한 건만 둡니다. 별칭 사전을 크게 만드는 것이 목적이 아니라, 별칭이 있는 경우와 없는 경우의 처리 경계를 확인하기 위한 최소 후보입니다.

In [6]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

ALIASES = {
    normalize_name('네이버'): normalize_name('NAVER'),
}

def exact_candidates(query, records):
    return [record for record in records if record['corp_name'] == query]

def normalized_candidates(query, records):
    normalized_query = ALIASES.get(normalize_name(query), normalize_name(query))
    return [
        record for record in records
        if normalized_query in {
            normalize_name(record['corp_name']),
            normalize_name(record['corp_eng_name']),
        }
    ]

def rank_candidates(candidates):
    return sorted(
        candidates,
        key=lambda record: (bool(record['stock_code'].strip()), record['modify_date']),
        reverse=True,
    )

def select_if_unique(candidates):
    return candidates[0] if len(candidates) == 1 else None

def select_listed_if_unique(candidates):
    listed_candidates = [record for record in candidates if record['stock_code'].strip()]
    if len(listed_candidates) == 1:
        return listed_candidates[0]
    return candidates[0] if len(candidates) == 1 else None

## 4. 동일 사례에서 두 후보 평가

정확 일치 기준선은 중복 후보를 자동 선택하지 않습니다. 정규화 후보도 상장 후보가 여럿이면 `보류`로 남겨 안전하지 않은 자동 선택을 피합니다.

In [7]:
WARMUP_ITERATIONS = 3
MEASURE_ITERATIONS = 20

def evaluate_strategy(strategy_name, candidate_function, selection_function):
    results = []
    latency_samples = []
    for case in DEVELOPMENT_CASES + SAFETY_CASES:
        for _ in range(WARMUP_ITERATIONS):
            selection_function(rank_candidates(candidate_function(case['질문 기업명'], corp_records)))

        case_latencies = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            candidates = rank_candidates(candidate_function(case['질문 기업명'], corp_records))
            selected = selection_function(candidates)
            case_latencies.append((time.perf_counter() - started_at) * 1_000)

        candidate_stock_codes = [record['stock_code'] for record in candidates if record['stock_code'].strip()]
        selected_stock_code = selected['stock_code'] if selected else None
        expected_stock_code = case['정답 종목코드']
        results.append({
            **case,
            '전략': strategy_name,
            '후보 수': len(candidates),
            '후보 종목코드': candidate_stock_codes,
            '선택 법인명': selected['corp_name'] if selected else None,
            '선택 종목코드': selected_stock_code,
            '정답 여부': selected_stock_code == expected_stock_code,
            '후보 포함 여부': expected_stock_code in candidate_stock_codes if expected_stock_code else not candidates,
            '반복 중앙 지연시간(ms)': median(case_latencies),
        })
        latency_samples.extend(case_latencies)
    return results, latency_samples

baseline_results, baseline_latencies = evaluate_strategy('정확 일치', exact_candidates, select_if_unique)
normalized_results, normalized_latencies = evaluate_strategy('정규화·별칭·상장 우선', normalized_candidates, select_listed_if_unique)
all_results = baseline_results + normalized_results

for row in all_results:
    print({
        key: row[key]
        for key in ['전략', '유형', '질문 기업명', '후보 수', '후보 종목코드', '선택 법인명', '선택 종목코드', '정답 여부']
    })

{'전략': '정확 일치', '유형': '정확 법인명', '질문 기업명': '삼성전자', '후보 수': 1, '후보 종목코드': ['005930'], '선택 법인명': '삼성전자', '선택 종목코드': '005930', '정답 여부': True}
{'전략': '정확 일치', '유형': '정확 법인명', '질문 기업명': 'SK하이닉스', '후보 수': 1, '후보 종목코드': ['000660'], '선택 법인명': 'SK하이닉스', '선택 종목코드': '000660', '정답 여부': True}
{'전략': '정확 일치', '유형': '공백 변형', '질문 기업명': 'SK 하이닉스', '후보 수': 0, '후보 종목코드': [], '선택 법인명': None, '선택 종목코드': None, '정답 여부': False}
{'전략': '정확 일치', '유형': '공백 변형', '질문 기업명': '현대 자동차', '후보 수': 0, '후보 종목코드': [], '선택 법인명': None, '선택 종목코드': None, '정답 여부': False}
{'전략': '정확 일치', '유형': '대소문자 변형', '질문 기업명': 'naver', '후보 수': 0, '후보 종목코드': [], '선택 법인명': None, '선택 종목코드': None, '정답 여부': False}
{'전략': '정확 일치', '유형': '별칭', '질문 기업명': '네이버', '후보 수': 0, '후보 종목코드': [], '선택 법인명': None, '선택 종목코드': None, '정답 여부': False}
{'전략': '정확 일치', '유형': '동일 법인명', '질문 기업명': '셀트리온', '후보 수': 2, '후보 종목코드': ['068270'], '선택 법인명': None, '선택 종목코드': None, '정답 여부': False}
{'전략': '정확 일치', '유형': '정확 법인명', '질문 기업명': '리노공업', '후보 수': 1, '후보 종목코드': ['058470'], '선택

In [8]:
def percentile(values, percentile_value):
    ordered = sorted(values)
    position = round((len(ordered) - 1) * percentile_value)
    return ordered[position]

def summarize_results(results, latencies):
    positive_rows = [row for row in results if row['정답 종목코드'] is not None]
    safety_rows = [row for row in results if row['정답 종목코드'] is None]
    selected = [row for row in results if row['선택 종목코드'] is not None]
    selected_positive = [row for row in positive_rows if row['선택 종목코드'] is not None]
    wrong_selected = [row for row in selected if not row['정답 여부']]
    return {
        '전략': results[0]['전략'],
        'Entity Accuracy': sum(row['정답 여부'] for row in positive_rows) / len(positive_rows),
        'Candidate Recall': sum(row['후보 포함 여부'] for row in positive_rows) / len(positive_rows),
        'Coverage': len(selected_positive) / len(positive_rows),
        'Unsafe Selection Rate': len(wrong_selected) / len(selected) if selected else 0.0,
        '보류 정확도': sum(row['정답 여부'] for row in safety_rows) / len(safety_rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': percentile(latencies, 0.95),
    }

metric_summary = [
    summarize_results(baseline_results, baseline_latencies),
    summarize_results(normalized_results, normalized_latencies),
]
for row in metric_summary:
    print({
        key: round(value, 4) if isinstance(value, float) else value
        for key, value in row.items()
    })

{'전략': '정확 일치', 'Entity Accuracy': 0.375, 'Candidate Recall': 0.5, 'Coverage': 0.375, 'Unsafe Selection Rate': 0.0, '보류 정확도': 1.0, 'p50 지연시간(ms)': 5.12, 'p95 지연시간(ms)': 6.4707}
{'전략': '정규화·별칭·상장 우선', 'Entity Accuracy': 1.0, 'Candidate Recall': 1.0, 'Coverage': 1.0, 'Unsafe Selection Rate': 0.0, '보류 정확도': 1.0, 'p50 지연시간(ms)': 207.1442, 'p95 지연시간(ms)': 228.3012}


## 5. 실행 후 해석

- 정규화 후보의 Entity Accuracy와 Candidate Recall이 기준선보다 높고 Unsafe Selection Rate가 늘지 않으면, 다음 후보로 유지합니다.
- Coverage만 높고 잘못 선택하는 사례가 있으면 채택하지 않습니다. 해당 사례는 보류 후 사용자 확인으로 처리해야 합니다.
- `셀트리온`처럼 복수 후보가 남으면 후보의 `corp_code`, `stock_code`, `modify_date`를 직접 확인합니다. 상장 종목이 하나면 선택할 수 있지만, 여럿이면 자동 선택하지 않습니다.
- 이 개발용 8건과 보류 1건만으로 최종 방식을 확정하지 않습니다. 결과 확인 뒤 별칭·동일 법인명·영문명·상장구분·실제 다상장 모호성 사례를 늘린 평가셋과 holdout 구성을 정합니다.
- `stock_code`는 기업코드 목록에서 상장 후보를 우선순위화하는 근거입니다. V1의 Y/K/N 상장 범위 최종 판정은 이후 `company.json`의 `corp_cls`를 함께 확인해 별도 검증합니다.